# Solution D2 · Diagnosing missingness mechanisms: what you can and cannot know

**Type**: Interpret

## Objective

For each column with missing values in the loans dataset, establish with statistical tests whether the missingness is compatible
with MCAR or depends on other variables (MAR), and say what the data **cannot** prove.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats

df = ds.loans()
with_missing = [c for c in df.columns if df[c].isna().any()]
df[with_missing].isna().mean().round(3)

## Commented solution

In [ ]:
def dependence_test(df, col):
    m = df[col].isna()
    p_values = {}
    for c in df.columns:
        if c == col:
            continue
        ok = df[c].notna()                                   # compare only where c is observed
        if pd.api.types.is_numeric_dtype(df[c]):
            p_values[c] = stats.mannwhitneyu(df.loc[ok & m, c], df.loc[ok & ~m, c]).pvalue
        else:
            p_values[c] = stats.chi2_contingency(pd.crosstab(m[ok], df.loc[ok, c]))[1]
    column = min(p_values, key=p_values.get)
    corrected_min_p = min(1.0, p_values[column] * len(p_values))    # Bonferroni
    return corrected_min_p, column

In [ ]:
rows = {}
for col in with_missing:
    p, by = dependence_test(df, col)
    rows[col] = {"missing_share": df[col].isna().mean(), "corrected_p": p, "depends_on": by}
table = pd.DataFrame(rows).T
mechanism = {col: ("MAR" if row.corrected_p < 0.001 else "MCAR") for col, row in table.iterrows()}
pd.options.display.float_format = "{:.3g}".format
table

In [ ]:
missing = df["income"].isna()
default_if_income_missing = df.loc[missing, "defaulted"].mean()
default_if_income_present = df.loc[~missing, "defaulted"].mean()
print(f"default rate: income missing {default_if_income_missing:.3f} | income present {default_if_income_present:.3f}")
true_false = {"a": False, "b": False, "c": True}

## Check

In [ ]:
with verify():
    p, by = dependence_test(df, "income")
    check(0 <= p <= 1 and by == "defaulted", "for 'income' the strongest dependence is with the target defaulted",
          "income is missing much more often among defaulters: the test on the target must have the minimum p-value")
    p2, by2 = dependence_test(df, "job_tenure")
    check(by2 == "contract_type" and p2 < 1e-6, "'job_tenure' depends on the contract type", "the missingness depends on contract_type")
    check(list(table.columns) == ["missing_share", "corrected_p", "depends_on"] and set(table.index) == set(with_missing),
          "table with one row per column and the three required columns", "columns: missing_share, corrected_p, depends_on")
    check(mechanism == {"age": "MCAR", "income": "MAR", "job_tenure": "MAR", "education": "MCAR", "occupation": "MCAR"},
          "mechanisms diagnosed correctly", "MAR where corrected_p < 0.001, MCAR elsewhere")
    check(default_if_income_missing > 3 * default_if_income_present, "whoever does not declare income defaults much more often (over 3 times)",
          "compare the default rate between rows with income missing and present")
    check(true_false == {"a": False, "b": False, "c": True}, "statements judged correctly",
          "a non-significant test proves nothing; MNAR is not detectable from the observed ones; the missingness is informative")
done()

## What you learned

- The tests correctly identify the dependencies: `income` is missing based on the target, `job_tenure` based on the contract
  type (whoever is unemployed has no tenure to report), and the other three columns are compatible with MCAR.
- **A non-significant test does not prove MCAR**: it only says that that test did not find a dependence. With few missing values or
  weak dependencies the tests have no power.
- **MNAR cannot be seen from the observed ones**: if the missingness depends on the missing value (whoever has a very high income does not
  declare it), the observed variables may show no dependence at all. It is a hypothesis about the world, not a result of the data:
  it is supported with knowledge of the collection process.
- The very fact that income is missing is information: 56% default among those who do not declare it against 15% among those who do.
  An imputer with `add_indicator=True` keeps it.

## Common pitfall

Concluding "MCAR, so I can drop the rows" after running an underpowered test. And the opposite: correcting the
p-values with Bonferroni and rejecting everything, when the real dependence is weak. In practice you combine the tests with
domain knowledge and check **empirically** whether the imputation choice changes the performance.

## Interview question

<details><summary>A colleague says: 'I checked, the missing data are MCAR, so I drop the rows'. How do you reply?</summary>

That MCAR cannot be verified: the tests can only detect dependencies on the observed variables, and a non-significant
result is not proof (it depends on the power). Besides, MNAR is indistinguishable from the observed ones. Dropping the rows
with many missing values also loses data and can introduce bias. I would compare dropping, simple imputation and imputation with
indicator in cross-validation on the real problem, and use knowledge of how the data are collected.

</details>